# __Homework 4:__ Practical analysis with BioPython

For the homework, you are going to extend the code from the analysis of our FASTQ file in lectures 8 and 9.
Recall that the FASTQ file contains reads from a real sequencing run of influenza virus HA and NA genes.

---
The __actual sequences__ are as follows:

    5'-[end of HA]-AGGCGGCCGC-[16 X N barcode]-3'
or 

    5'-[end of NA]-AGGCGGCCGC-[16 X N barcode]-3'
---


__The end of NA is__ `...CACGATAGATAAATAATAGTGCACCAT`
    
__The end of HA is__ `...CCGGATTTGCATATAATGATGCACCAT`

---    

    
The __sequencing reads__ from the reverse end of the molecules (in 5'>3' orientation), so the sequencing reads are as follows:

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of HA]-3'
or

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of NA]-3'

---   
    
The reads can originate from **either** HA or NA, and that will be distinguished by the most 3' end of the read.
But in our example exercise in class, we did not distinguish among reads matching to HA and NA, as we didn't even look far enough into the read to tell the identity.

For the homework, your goal is to write code that extends the material from lectures 8 and 9 to also distinguish between HA and NA.
This homework can be completed almost entirely by re-using code from lecture 9. You will need to set up your analysis to do the following:
 1. Get the reverse complement of each read.
 2. Determine if it matches the expected pattern for HA and NA, and if so which one.
 3. If it matches, extract the barcode and add it to a dictionary to keep track of counts.
 4. Determine the number and distribution of barcodes for HA and NA separately.

Please include code to address each of the following questions. Please include code comments to explain what your code is attempting to accomplish. Don't forget to include references to the sources you used to obtain your answer, including your classmates (if you are working in groups).  

1. How many reads map to HA, and how many reads map to NA?

   Please __include at least two test cases__ showing that your HA/NA classifier works. You can copy the `run_tests` helper from lecture 9, or use `assert` statements - either is fine. These test cases are part of the points for this question.


In [ ]:
# importing programs needed

import re
import Bio.Seq
import Bio.SeqIO

In [ ]:
# shows reverse complement of the genes

print(Bio.Seq.Seq("CACGATAGATAAATAATAGTGCACCAT").reverse_complement())
print(Bio.Seq.Seq("CCGGATTTGCATATAATGATGCACCAT").reverse_complement())


ATGGTGCACTATTATTTATCTATCGTG
ATGGTGCATCATTATATGCAAATCCGG


In [ ]:
# function for running a list of tests
# provided in lecture 09

def run_tests(parser_function, test_cases):
    """Run a parser against a list of test cases and report the results.

    Args:
        parser_function: The function to test. It takes a sequence and returns a barcode
            string, or None.
        test_cases: A list of (sequence, expected result, label) tuples.

    Returns:
        The number of test cases that failed.
    """
    n_failed = 0

    for seq, expected, label in test_cases:
        result = parser_function(seq)
        if result == expected:
            print(f"PASS  {label}")
        else:
            n_failed += 1
            print(f"FAIL  {label}: {seq!r} gave {result!r}, expected {expected!r}")

    return f"{n_failed} tests failed"

In [18]:
# defining variables - the end of each gene(forward orientation) and the common linker sequence
# writing the parsing function

NA = "CACGATAGATAAATAATAGTGCACCAT"
HA = "CCGGATTTGCATATAATGATGCACCAT"
linker = "AGGCGGCCGC"

def read_barcode(seqread, bclen=16, upstream=linker):
    """Identify a barcode with a known upstream sequence.

    Args:
        seqread: Nucleotide sequence read (a Seq object or a string) matching
            in reverse orientation.
        bclen: Length of the barcode.
        upstream: Sequence expected immediately upstream of the barcode (forward orientation).

    Returns:
        The barcode in the forward orientation, as an upper-case string, or
        None if the read does not match the expected design.
    """
    seqread_upper = str(seqread).upper()
    seqread_original = str(Bio.Seq.Seq(seqread_upper).reverse_complement())

    barcode_pattern = re.compile(f"{upstream}(?P<barcode>[ACTGN]{{{bclen}}})$")

    match = barcode_pattern.search(seqread_original)
    if match:
        return match.group("barcode") 
    else:
        return None

In [ ]:
# flips the read back to the original orientation to find the HA and NA endings
# differentiating between HA and NA genes using the ending

def classify_read(seqread):
    """Classify a read as HA or NA based on its gene ending.

    Returns:
        "HA", "NA", or None if the read matches neither ending.
    """
    seqread_original = str(Bio.Seq.Seq(str(seqread).upper()).reverse_complement())
    if HA in seqread_original:
        return "HA"
    elif NA in seqread_original:
        return "NA"
    else:
        return None

In [ ]:
# creating sample string and applying it to test cases
# ensuring the code is running properly
reverse_complement_ha = str(Bio.Seq.Seq(HA + linker + "A"*16).reverse_complement())
reverse_complement_na = str(Bio.Seq.Seq(NA + linker + "A"*16).reverse_complement())

test_cases = [
    (reverse_complement_ha, "HA", "positive: HA read"),
    (reverse_complement_na, "NA", "positive: NA read"),
    ("ACGTACGTACGT", None, "negative: neither gene"),
    (reverse_complement_ha.lower(), "HA", "positive: lowercase, but still HA read"),
]

run_tests(classify_read, test_cases)

PASS  positive: HA read
PASS  positive: NA read
PASS  negative: neither gene
PASS  positive: lowercase, but still HA read


'0 tests failed'

In [ ]:
# loading reads from FASTQ file into a list called seqreads
# counting the reads per gene 
# classifying each read and adding one to each count
seqreads = list(Bio.SeqIO.parse("barcodes_R1.fastq", format="fastq"))

counts = {"HA": 0, "NA": 0, None: 0}

for read in seqreads:
    gene = classify_read(read.seq)
    counts[gene] += 1

print(counts)

{'HA': 5409, 'NA': 4122, None: 469}


5409 reads map to HA and 4122 map to NA (469 reads matched neither) 

Sources: lecture 08/09 for functions/programs/code and Claude AI for understanding errors.

2. How many HA sequences did not have a valid barcode? Also anwer the same question for NA.

In [ ]:
# count reads that were classified at HA or NA but did not have a valid barcode
# the gene is decided by the HA or NA ending alone then checks if HA/NA + barcode is included

no_barcode = {"HA": 0, "NA": 0}

for read in seqreads:
    gene = classify_read(read.seq)
    if gene == "HA":
        barcode = read_barcode(read.seq, upstream=HA + linker)
        if barcode is None:
            no_barcode["HA"] += 1
    elif gene == "NA":
        barcode = read_barcode(read.seq, upstream=NA + linker)
        if barcode is None:
            no_barcode["NA"] += 1

print(no_barcode)

{'HA': 163, 'NA': 212}


There are 163 HA sequences and 212 NA sequences that do not have a valid barcode.

Sources: lecture 09 + previous lectures on Python

3. What is the HA barcode with the most counts (and how many counts)? Also answer the same question for NA.

    _Hint: you will need to find the key associated with the maximum value in your dictionary. There are many ways to do this._

In [ ]:
# created a dictionary for each gene where the key is the barcode and the value is number of reads that have that barcode
# genes come from classify_read and are verified using read_barcode
# skip reads with no valid barcode
# if the barcode is the same as another add one, if its new count it as 1

HA_barcodes = {}
NA_barcodes = {}

for read in seqreads:
    gene = classify_read(read.seq)
    if gene == "HA":
        barcode = read_barcode(read.seq, upstream=HA + linker)
        if barcode is not None:
                if barcode in HA_barcodes:
                    HA_barcodes[barcode] += 1
                else:
                    HA_barcodes[barcode] = 1
    elif gene == "NA":
        barcode = read_barcode(read.seq, upstream=NA + linker)
        if barcode is not None:
                if barcode in NA_barcodes:
                    NA_barcodes[barcode] += 1
                else:
                    NA_barcodes[barcode] = 1

print("number of unique HA_barcodes:", len(HA_barcodes))
print("number of unique NA_barcodes:", len(NA_barcodes))

number of unique HA_barcodes: 281
number of unique NA_barcodes: 138


In [ ]:
# compare barcodes by their counts and print the most common HA or NA gene

most_HA = max(HA_barcodes, key=HA_barcodes.get)
print("the most common HA barcode is:", most_HA, "with", HA_barcodes[most_HA], "counts")

most_NA = max(NA_barcodes, key=NA_barcodes.get)
print("the most common NA barcode is:", most_NA, "with", NA_barcodes[most_NA], "counts")

the most common HA barcode is: CCCGACCCGACATTAA with 155 counts
the most common NA barcode is: ACCAGTTCTCCCCGGG with 152 counts


The most common HA barcode is CCCGACCCGACATTAA with 155 counts
The most common NA barcode is ACCAGTTCTCCCCGGG with 152 counts